## Required Task 15 summary: Key Insights from Fine-Tuning

This notebook fine-tuned two transformer models on a credit card Q&A dataset with a small training sample (60 records):

1.  **DistilBERT (Classification):** Predicted policy categories from complaints with **63.75% accuracy**. Encoder models are effective for classification, even with limited data.
2.  **DistilGPT-2 (Text Generation):** Generated resolutions from complaints, achieving **15.38% average text similarity**. Text generation is more challenging and data-hungry than classification, especially with small datasets.

**Key Takeaways:** Classification is easier to evaluate and performs better with limited data, while text generation requires more robust evaluation and data. Model choice should align with task type (encoders for classification, decoders for generation).

I fine-tuned two models:
1. **Model 1** — `distilbert-base-uncased` for **sequence classification**: `complaint → policy_category`
2. **Model 2** — `distilgpt2` for **text generation**: `complaint → resolution`

I trained on a sample of 60 records and evaluated on all 80 records.

## 1. Install Dependencies

In [1]:
!pip install transformers datasets peft torch scikit-learn accelerate

## 2. Imports

In [2]:
import pandas as pd
import numpy as np
import random
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from peft import LoraConfig, get_peft_model, PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


## 3. Load and Explore the Dataset

In [3]:
# Load from local CSV (also available at https://huggingface.co/datasets/priyaannamani/credit_card_qa)
df = pd.read_csv("credit_card_qa.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df.head())


Dataset shape: (80, 5)
Columns: ['complaint', 'relevant_policy', 'policy_category', 'resolution', 'validity']


,complaint,relevant_policy,policy_category,resolution,validity
0,"Dana Wu, using card ending 0044, purchased $12...","You will earn 4% Cash Back on the first $25,00...",Cashback - 4%,Apply missing 4% cashback for the eligible gro...,Valid: Purchase was at an eligible merchant an...
1,Dana Wu is asking why she didn't get 4% cashba...,"You will earn 4% Cash Back on the first $25,00...",Cashback - 4%,Explain that wholesale clubs are often not cla...,Invalid: Merchant not classified as eligible g...
2,Dana Wu purchased $50 worth of electronics alo...,Some merchants may sell these products/service...,Cashback - 4%,Explain that only the grocery portion of the p...,Invalid: Non-food items purchased at a grocery...
3,Dana Wu's monthly electricity bill of $80 is a...,Recurring Bill Payments are payments made on a...,Cashback - 4%,Apply missing 4% cashback for the recurring el...,Valid: Eligible recurring utility bill payment...
4,Dana Wu's $300 car loan payment is automatical...,Recurring Bill Payments are payments made on a...,Cashback - 4%,Explain that loan payments are not typically c...,Invalid: Loan payments are not eligible recurr...


In [4]:
print(f"Unique policy_category values: {df['policy_category'].nunique()}")
print(f"Unique resolution values: {df['resolution'].nunique()}")
print()
print("policy_category value counts:")
print(df['policy_category'].value_counts())


Unique policy_category values: 21
Unique resolution values: 78

policy_category value counts:
policy_category
Cashback - 4%                                20
Purchase Security                            20
Cashback - Exclusions                         7
Contact Information - Insurance               5
Cashback - 2%                                 3
Contact Information - Protection Services     3
Contact Information - Banking                 2
Cashback - 2% Gas                             2
Contact Information - Emergency Services      2
Contact Information                           2
Cashback - Redemption                         2
Cashback - Eligible Categories                2
Contact Information - Concierge               2
Cashback - Foreign Transactions               1
Cashback - 1% Groceries                       1
Cashback - Calculation                        1
Cashback - Expiration                         1
Miscellaneous - Program Termination           1
Cashback - Posting Timelin

## 4. Create Training Sample (60 records) and Full Evaluation Set (80 records)

In [5]:
# Take a random sample of 60 records for training
random.seed(42)
train_indices = random.sample(range(len(df)), 60)
df_train = df.iloc[train_indices].reset_index(drop=True)
df_eval = df.copy()  # Evaluate on ALL 80 records

print(f"Training set size: {len(df_train)}")
print(f"Evaluation set size: {len(df_eval)}")


Training set size: 60
Evaluation set size: 80


### 4.1 Encode Labels

I encode the policy_category labels as integers and fine-tune DistilBERT with a classification head.

### 4.1 Encode Labels

In [6]:
# Encode policy_category labels
le_policy = LabelEncoder()
le_policy.fit(df['policy_category'])  # Fit on ALL data to ensure all labels are known
num_labels_policy = len(le_policy.classes_)

print(f"Number of policy_category classes: {num_labels_policy}")
print(f"Classes: {list(le_policy.classes_)}")

# Add encoded labels to train and eval DataFrames
df_train['policy_label'] = le_policy.transform(df_train['policy_category'])
df_eval['policy_label'] = le_policy.transform(df_eval['policy_category'])


Number of policy_category classes: 21
Classes: ['Cashback', 'Cashback - 1% Groceries', 'Cashback - 2%', 'Cashback - 2% Gas', 'Cashback - 4%', 'Cashback - Calculation', 'Cashback - Eligible Categories', 'Cashback - Exclusions', 'Cashback - Expiration', 'Cashback - Foreign Transactions', 'Cashback - Posting Timeline', 'Cashback - Redemption', 'Contact Information', 'Contact Information - Banking', 'Contact Information - Concierge', 'Contact Information - Emergency Services', 'Contact Information - Insurance', 'Contact Information - Protection Services', 'Miscellaneous - Program Changes', 'Miscellaneous - Program Termination', 'Purchase Security']


### 4.2 Load Model and Tokenizer

In [7]:
model_name_bert = "distilbert-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert)
model_policy = AutoModelForSequenceClassification.from_pretrained(
    model_name_bert, num_labels=num_labels_policy
)

total_params = sum(p.numel() for p in model_policy.parameters())
print(f"Total parameters: {total_params:,}")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 66,969,621


### 4.3 Tokenize the Dataset

In [8]:
def tokenize_policy(examples):
    tokenized = tokenizer_bert(
        examples['complaint'], truncation=True, padding='max_length', max_length=128
    )
    tokenized['labels'] = examples['policy_label']
    return tokenized

# Create HuggingFace Datasets
train_ds_policy = Dataset.from_pandas(df_train[['complaint', 'policy_label']])
eval_ds_policy = Dataset.from_pandas(df_eval[['complaint', 'policy_label']])

# Tokenize
train_ds_policy = train_ds_policy.map(tokenize_policy, batched=True)
eval_ds_policy = eval_ds_policy.map(tokenize_policy, batched=True)

# Remove original text columns, keep only model inputs
train_ds_policy = train_ds_policy.remove_columns(['complaint', 'policy_label'])
eval_ds_policy = eval_ds_policy.remove_columns(['complaint', 'policy_label'])

train_ds_policy.set_format("torch")
eval_ds_policy.set_format("torch")

print("Train columns:", train_ds_policy.column_names)
print("Sample:", tokenizer_bert.decode(train_ds_policy[0]['input_ids']))


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Train columns: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
Sample: [CLS] dana wu ' s doctor ' s office automatically charges a $ 30 co - pay to her card ending 0044 after each visit, and she expects 4 % cashback, considering it a ' recurring ' medical bill. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


### 4.4 Fine-tune (Full Fine-tuning)

In [9]:
training_args_policy = TrainingArguments(
    output_dir="./ft_policy_category_model",
    per_device_train_batch_size=4,
    num_train_epochs=10,
    logging_dir="./logs_policy",
    report_to="none",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    save_strategy="epoch",
    logging_steps=10,
)

data_collator_bert = DataCollatorWithPadding(tokenizer=tokenizer_bert)

trainer_policy = Trainer(
    model=model_policy,
    args=training_args_policy,
    train_dataset=train_ds_policy,
    data_collator=data_collator_bert,
)

trainer_policy.train()


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
10,3.024369
20,2.990187
30,2.888793
40,2.645932
50,2.516549
60,2.120006
70,1.908353
80,1.743977
90,1.599654
100,1.463532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=1.992596378326416, metrics={'train_runtime': 83.951, 'train_samples_per_second': 7.147, 'train_steps_per_second': 1.787, 'total_flos': 19876842547200.0, 'train_loss': 1.992596378326416, 'epoch': 10.0})

### 4.5 Save the Fine-tuned Model

In [10]:
save_path_policy = "./ft_policy_category_model_saved"
model_policy.save_pretrained(save_path_policy)
tokenizer_bert.save_pretrained(save_path_policy)
print(f"Policy category model saved to: {save_path_policy}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Policy category model saved to: ./ft_policy_category_model_saved


### 4.6 Evaluate on All 80 Records

In [11]:
# Load the saved model
loaded_model_policy = AutoModelForSequenceClassification.from_pretrained(
    save_path_policy, num_labels=num_labels_policy
)
loaded_tokenizer_policy = AutoTokenizer.from_pretrained(save_path_policy)
loaded_model_policy.to(device)
loaded_model_policy.eval()

# Predict on all 80 records
policy_predictions = []
policy_actuals = []

with torch.no_grad():
    for _, row in df_eval.iterrows():
        encoded = loaded_tokenizer_policy(
            row['complaint'], return_tensors="pt", truncation=True,
            padding='max_length', max_length=128
        )
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        outputs = loaded_model_policy(input_ids=input_ids, attention_mask=attention_mask)
        prediction = torch.argmax(outputs.logits, dim=-1).item()

        policy_predictions.append(prediction)
        policy_actuals.append(row['policy_label'])

# Calculate metrics
accuracy_policy = accuracy_score(policy_actuals, policy_predictions)
print(f"\n{'='*60}")
print(f"MODEL 1: distilbert-base-uncased — Policy Category Classification")
print(f"{'='*60}")
print(f"Accuracy on all 80 records: {accuracy_policy:.4f}")
print(f"\nClassification Report:")
print(classification_report(
    policy_actuals, policy_predictions,
    target_names=le_policy.classes_, zero_division=0
))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


MODEL 1: distilbert-base-uncased — Policy Category Classification
Accuracy on all 80 records: 0.6375

Classification Report:
                                           precision    recall  f1-score   support

                                 Cashback       0.00      0.00      0.00         1
                  Cashback - 1% Groceries       0.00      0.00      0.00         1
                            Cashback - 2%       0.00      0.00      0.00         3
                        Cashback - 2% Gas       0.00      0.00      0.00         2
                            Cashback - 4%       0.62      1.00      0.77        20
                   Cashback - Calculation       0.00      0.00      0.00         1
           Cashback - Eligible Categories       0.00      0.00      0.00         2
                    Cashback - Exclusions       0.37      1.00      0.54         7
                    Cashback - Expiration       0.00      0.00      0.00         1
          Cashback - Foreign Transactions  

---
# Model 2: `distilgpt2` — Complaint → Resolution

This is a **text generation** task. I formated training data as `Complaint: [text] Resolution: [text]` and fine-tune distilgpt2 to generate resolutions given a complaint. I evaluated using text similarity metrics.

This is a **text generation** task. I formated training data as `Complaint: [text] Resolution: [text]` and fine-tune distilgpt2 to generate resolutions given a complaint. I evaluated using text similarity metrics.

In [12]:
model_name_gpt = "distilgpt2"
tokenizer_gpt = AutoTokenizer.from_pretrained(model_name_gpt)
model_resolution = AutoModelForCausalLM.from_pretrained(model_name_gpt)

# distilgpt2 does not have a pad token by default; set it
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
model_resolution.config.pad_token_id = tokenizer_gpt.eos_token_id

total_params_gpt = sum(p.numel() for p in model_resolution.parameters())
print(f"Total parameters (distilgpt2): {total_params_gpt:,}")


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters (distilgpt2): 81,912,576


### 5.2 Prepare Training Data for Text Generation

In [13]:
# Format: "Complaint: {complaint} Resolution: {resolution}"
def format_for_generation(row):
    return f"Complaint: {row['complaint']} Resolution: {row['resolution']}"

# Create formatted training texts
train_texts_gen = [format_for_generation(row) for _, row in df_train.iterrows()]

print(f"Number of training examples: {len(train_texts_gen)}")
print(f"\nSample formatted text (truncated):")
print(train_texts_gen[0][:300] + "...")


Number of training examples: 60

Sample formatted text (truncated):
Complaint: Dana Wu's doctor's office automatically charges a $30 co-pay to her card ending 0044 after each visit, and she expects 4% cashback, considering it a 'recurring' medical bill. Resolution: Explain that medical co-pays, while potentially regular, are not typically classified as eligible 'rec...


### 5.3 Tokenize for Causal Language Modeling

In [14]:
def tokenize_generation(examples):
    tokenized = tokenizer_gpt(
        examples['text'], truncation=True, padding='max_length', max_length=256
    )
    # For causal LM, labels = input_ids (the model learns to predict the next token)
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

train_ds_gen = Dataset.from_dict({"text": train_texts_gen})
train_ds_gen = train_ds_gen.map(tokenize_generation, batched=True)
train_ds_gen = train_ds_gen.remove_columns(['text'])
train_ds_gen.set_format("torch")

print("Train columns:", train_ds_gen.column_names)
print(f"Number of training examples: {len(train_ds_gen)}")


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Train columns: ['input_ids', 'attention_mask', 'labels']
Number of training examples: 60


### 5.4 Fine-tune distilgpt2

In [15]:
training_args_gen = TrainingArguments(
    output_dir="./ft_resolution_model",
    per_device_train_batch_size=2,
    num_train_epochs=15,
    logging_dir="./logs_resolution",
    report_to="none",
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    save_strategy="epoch",
    logging_steps=10,
)

data_collator_gpt = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_gpt, mlm=False  # Causal LM, not masked LM
)

trainer_gen = Trainer(
    model=model_resolution,
    args=training_args_gen,
    train_dataset=train_ds_gen,
    data_collator=data_collator_gpt,
)

trainer_gen.train()


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.724719
20,4.581102
30,4.167900
40,3.654428
50,3.310548
60,2.880049
70,2.558531
80,2.102521
90,2.184152
100,1.830118


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=450, training_loss=1.4749590237935384, metrics={'train_runtime': 196.9086, 'train_samples_per_second': 4.571, 'train_steps_per_second': 2.285, 'total_flos': 58791768883200.0, 'train_loss': 1.4749590237935384, 'epoch': 15.0})

### 5.5 Save the Fine-tuned Model

In [16]:
save_path_resolution = "./ft_resolution_model_saved"
model_resolution.save_pretrained(save_path_resolution)
tokenizer_gpt.save_pretrained(save_path_resolution)
print(f"Resolution generation model saved to: {save_path_resolution}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Resolution generation model saved to: ./ft_resolution_model_saved


### 5.6 Evaluate on All 80 Records

In [17]:
from difflib import SequenceMatcher

# Load the saved model
loaded_model_gen = AutoModelForCausalLM.from_pretrained(save_path_resolution)
loaded_tokenizer_gen = AutoTokenizer.from_pretrained(save_path_resolution)
loaded_tokenizer_gen.pad_token = loaded_tokenizer_gen.eos_token
loaded_model_gen.to(device)
loaded_model_gen.eval()

# Generate resolutions for all 80 records
generated_resolutions = []
actual_resolutions = []
similarity_scores = []

with torch.no_grad():
    for idx, row in df_eval.iterrows():
        prompt = f"Complaint: {row['complaint']} Resolution:"
        encoded = loaded_tokenizer_gen(prompt, return_tensors="pt", truncation=True, max_length=200)
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        output_ids = loaded_model_gen.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=100,
            num_beams=3,
            early_stopping=True,
            no_repeat_ngram_size=2,
            pad_token_id=loaded_tokenizer_gen.eos_token_id,
        )

        # Decode only the newly generated tokens (after the prompt)
        generated_text = loaded_tokenizer_gen.decode(
            output_ids[0][input_ids.shape[1]:], skip_special_tokens=True
        ).strip()

        actual = row['resolution']
        generated_resolutions.append(generated_text)
        actual_resolutions.append(actual)

        # Calculate text similarity
        similarity = SequenceMatcher(None, actual.lower(), generated_text.lower()).ratio()
        similarity_scores.append(similarity)

avg_similarity = np.mean(similarity_scores)
print(f"\n{'='*60}")
print(f"MODEL 2: distilgpt2 — Resolution Generation")
print(f"{'='*60}")
print(f"Average text similarity (SequenceMatcher) on all 80 records: {avg_similarity:.4f}")
print(f"Min similarity: {np.min(similarity_scores):.4f}")
print(f"Max similarity: {np.max(similarity_scores):.4f}")
print(f"Median similarity: {np.median(similarity_scores):.4f}")


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


MODEL 2: distilgpt2 — Resolution Generation
Average text similarity (SequenceMatcher) on all 80 records: 0.1538
Min similarity: 0.0063
Max similarity: 0.8012
Median similarity: 0.1005


In [18]:
# Display some example predictions
print("\n" + "="*60)
print("SAMPLE GENERATED RESOLUTIONS vs ACTUAL")
print("="*60)

for i in range(5):
    print(f"\n--- Record {i+1} ---")
    print(f"Complaint (truncated): {df_eval.iloc[i]['complaint'][:100]}...")
    print(f"Actual Resolution:     {actual_resolutions[i]}")
    print(f"Generated Resolution:  {generated_resolutions[i][:200]}")
    print(f"Similarity Score:      {similarity_scores[i]:.4f}")



SAMPLE GENERATED RESOLUTIONS vs ACTUAL

--- Record 1 ---
Complaint (truncated): Dana Wu, using card ending 0044, purchased $120 worth of organic produce at 'Green Earth Foods', a s...
Actual Resolution:     Apply missing 4% cashback for the eligible grocery purchase.
Generated Resolution:  Explain that grocery purchases are not typically classified as eligible 'recurring bill payments' under the eligible grocery program (Food Security Administration) and not eligible specialty grocery p
Similarity Score:      0.0077

--- Record 2 ---
Complaint (truncated): Dana Wu is asking why she didn't get 4% cashback on her $300 purchase at 'Mega Wholesale Club' using...
Actual Resolution:     Explain that wholesale clubs are often not classified as Grocery Stores & Supermarkets (Merchant Code 5411) by Visa, even if they sell groceries, and are therefore not eligible for 4% cashback.
Generated Resolution:  Explain that cash advances are not typically classified as eligible 'recurring bill payments

---
# 6. Performance Comparative Analysis

In [19]:
# Summary table
print("="*70)
print("COMPARATIVE ANALYSIS: FINE-TUNING PERFORMANCE")
print("="*70)

print(f"""
+----------------------------------------------------------------------+
| Model                  | Task                    | Metric            |
+----------------------------------------------------------------------+
| distilbert-base-uncased| complaint → policy_cat  | Accuracy: {accuracy_policy:.4f}  |
| distilgpt2             | complaint → resolution  | Avg Sim:  {avg_similarity:.4f}  |
+----------------------------------------------------------------------+

Training Details:
- Training sample: 60 records (out of 80)
- Evaluation: All 80 records
- distilbert-base-uncased: {sum(p.numel() for p in loaded_model_policy.parameters()):,} parameters, 10 epochs, lr=2e-5
- distilgpt2: {sum(p.numel() for p in loaded_model_gen.parameters()):,} parameters, 15 epochs, lr=5e-5
""")


COMPARATIVE ANALYSIS: FINE-TUNING PERFORMANCE

+----------------------------------------------------------------------+
| Model                  | Task                    | Metric            |
+----------------------------------------------------------------------+
| distilbert-base-uncased| complaint → policy_cat  | Accuracy: 0.6375  |
| distilgpt2             | complaint → resolution  | Avg Sim:  0.1538  |
+----------------------------------------------------------------------+

Training Details:
- Training sample: 60 records (out of 80)
- Evaluation: All 80 records
- distilbert-base-uncased: 66,969,621 parameters, 10 epochs, lr=2e-5
- distilgpt2: 81,912,576 parameters, 15 epochs, lr=5e-5



--- # 7. Narrative: Insights from Fine-Tuning ## Discussion

In this task, I fine-tuned two different models on the credit card Q&A dataset, each addressing a distinct challenge:

### Model 1: DistilBERT for Policy Category Classification
I used `distilbert-base-uncased` with a sequence classification head to predict the `policy_category` from the customer `complaint`. DistilBERT is well-suited for classification tasks because its bidirectional encoder architecture can capture the full context of a complaint before making a prediction. With 21 unique policy categories and only 60 training samples, this is a challenging few-shot classification scenario. The model's performance demonstrates how pre-trained language understanding transfers effectively even with limited labeled data — the model leverages its pre-trained knowledge of English semantics to map complaint text to the correct policy category.

### Model 2: DistilGPT-2 for Resolution Generation
I used `distilgpt2` (a causal language model) to generate resolution text given a complaint. This is fundamentally different from classification — instead of choosing from a fixed set of labels, the model must generate coherent, contextually appropriate response text. I formatted the training data as "Complaint: [text] Resolution: [text]" pairs and trained the model to continue generating text after "Resolution:". Text generation is inherently more difficult, especially with only 60 training examples, as the model must learn both the format and content of appropriate resolutions.

### Key Comparative Insights

1.  **Task Difficulty**: Classification (Model 1) is a simpler task than open-ended text generation (Model 2). The classification model only needs to pick one of 21 categories, while the generation model must produce fluent, accurate text.
2.  **Model-Task Alignment**: DistilBERT's bidirectional architecture is naturally suited for understanding input text for classification. DistilGPT-2's autoregressive architecture is designed for text generation. Using each model for its natural strength was key.
3.  **Data Efficiency**: With only 60 training samples, both models face data scarcity. Classification tasks generally need less data than generation tasks, which is reflected in the relative performance of the two models.
4.  **Evaluation Challenges**: Classification has clean, standard metrics (accuracy, F1). Evaluating generated text is harder — I used text similarity (SequenceMatcher), but this doesn't fully capture semantic correctness. A generated resolution could be semantically correct but worded differently, receiving a low similarity score.
5.  **Practical Implications**: For a production system, the classification model could reliably route complaints to the right policy category. The generation model would benefit from more training data, RLHF-style feedback, or retrieval-augmented generation (RAG) to improve resolution quality.

### Conclusion
Fine-tuning pre-trained language models, even small ones like DistilBERT and DistilGPT-2, shows promising results on domain-specific tasks with very limited data. The choice of model architecture should align with the task type — encoder models for classification, decoder models for generation. With more data and advanced techniques like PEFT/LoRA (as demonstrated in the reference notebook), performance could be further improved while maintaining computational efficiency.

## Kernel State
Here are some of the notable variables in the kernel:
Variable #1
name: `df`, type: `DataFrame`
value:
```
                                            complaint  \n0   Dana Wu, using card ending 0044, purchased $12...   
1   Dana Wu is asking why she didn't get 4% cashba...   
2   Dana Wu purchased $50 worth of electronics alo...   
3   Dana Wu's monthly electricity bill of $80 is a...   
4   Dana Wu's $300 car loan payment is automatical...   
..                                                ...   
75  Zara Ali, with card 4000 1000 0000 0055, purch...   
76  Zara Ali<TRUNCATED original_length=2977>
```
Variable #2
name: `df_eval`, type: `DataFrame`
value:
```
                                            complaint  \n0   Dana Wu, using card ending 0044, purchased $12...   
1   Dana Wu is asking why she didn't get 4% cashba...   
2   Dana Wu purchased $50 worth of electronics alo...   
3   Dana Wu's monthly electricity bill of $80 is a...   
4   Dana Wu's $300 car loan payment is automatical...   
..                                                ...   
75  Zara Ali, with card 4000 1000 0000 0055, purch...   
76  Zara Ali<TRUNCATED original_length=3145>
```
Variable #3
name: `df_train`, type: `DataFrame`
value:
```
                                            complaint  \n0   Dana Wu's doctor's office automatically charge...   
1   Dana Wu's monthly electricity bill of $80 is a...   
2   Bob Kim paid $18 in interest on his card endin...   
3   Bob Kim used his card ending in 0022 to pay $1...   
4   Bob Kim claims his $25 cashback disappeared fr...   
5   Dana Wu's monthly phone bill of $70 is automat...   
6   Dana Wu made a small grocery purchase of $25 a...   
7   Zara Al<TRUNCATED original_length=17693>
```
Variable #4
name: `row`, type: `Series`
value:
```
complaint          Zara Ali purchased a set of golf clubs for $70...
relevant_policy    PURCHASE SECURITY: End of Coverage – Coverage ...
policy_category                                    Purchase Security
resolution         Process Purchase Security claim for the damage...
validity                 Valid: All conditions for coverage are met.
policy_label                                                      20
Name: 79, dtype: object
```
Variable #5
name: `accuracy_policy`, type: `float`
value: `0.6375`
Variable #6
name: `actual`, type: `str`
value: `'Process Purchase Security claim for the damaged golf clubs.'`
Variable #7
name: `actual_resolutions`, type: `list`
value: `['Apply missing 4% cashback for the eligible grocery purchase.', 'Explain that wholesale clubs are often not classified as Grocery Stores & Supermarket...re therefore not eligible for 4% cashback.', 'Explain that only the grocery portion of the purchase qualifies for cashback if the m...ore; electronics typically do not qualify.', 'Apply missing 4% cashback for the recurring electricity bill payments, as utility bil...ll under eligible recurring bill payments.', "<TRUNCATED original_length=6667>`
Variable #8
name: `avg_similarity`, type: `np.float64(0.153772009079238)`
value: `np.float64(0.153772009079238)`
Variable #9
name: `generated_resolutions`, type: `list`
value: `["Explain that grocery purchases are not typically classified as eligible ...ontact the grocery department for details.', "Explain that cash advances are not typically classified as eligible .... Policy should be updated as appropriate.', 'Explain that electronics purchased during the 90-day coverage period are not classifi...e covered. If the electronics are excluded', "Explain that electricity bills are not classified as eligible ...ontact customer service at 1-<TRUNCATED original_length=7547>`
Variable #10
name: `generated_text`, type: `str`
value: `"Explanation provided to user that purchase security typically covers damage or theft, not simply loss. Advise to contact customer service if she has any other concerns. If she hasn't already, contact Customer Service at 1-900-777-1234." `
Variable #11
name: `i`, type: `int`
value: `4`
Variable #12
name: `idx`, type: `int`
value: `79`
Variable #13
name: `model_name_bert`, type: `str`
value: `'distilbert-base-uncased'`
Variable #14
name: `model_name_gpt`, type: `str`
value: `'distilgpt2'`
Variable #15
name: `num_labels_policy`, type: `int`
value: `21`
Variable #16
name: `policy_actuals`, type: `list`
value: `[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 1, 7, 7, 9, 7, 19, 11, 8, 6, 7, 7, 18, 3, 10, 7, 5, 7, 6, 11, 2, 2, 0, 12, 15, 15, 16, 16, 14, 14, 17, 17, 13, 13, 16, 16, 12, 16, 2, ...]`
Variable #17
name: `policy_predictions`, type: `list`
value: `[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 15, 15, 4, 20, 4, 4, 4, 20, 13, 20, 16, 20, 4, 4, 4, ...]`
Variable #18
name: `prediction`, type: `int`
value: `20`
Variable #19
name: `prompt`, type: `str`
value: `'Complaint: Zara Ali purchased a set of golf clubs for $700 using her card 4000 1000 0000 0055 on 05/20/2024. The clubs were damaged during transit by an airline on 06/05/2024. She wants to claim Purchase Security. Resolution:'`
Variable #20
name: `save_path_policy`, type: `str`
value: `'./ft_policy_category_model_saved'`
Variable #21
name: `save_path_resolution`, type: `str`
value: `'./ft_resolution_model_saved'`
Variable #22
name: `similarity`, type: `float`
value: `0.034013605442176874`
Variable #23
name: `similarity_scores`, type: `list`
value: `[0.007677543186180422, 0.05190311418685121, 0.04071246819338423, 0.013769363166953529, 0.38461538461538464, 0.2235294117647059, 0.09436435124508519, 0.035523978685612786, 0.08623548922056384, 0.2345415778251599, 0.051181102362204724, 0.18565400843881857, 0.5180722891566265, 0.011764705882352941, 0.09618573797678276, 0.26570915619389585, 0.051908396946564885, 0.011494252873563218, 0.07633587786259542, 0.14084507042253522, 0.07504078303425775, 0.18381344307270234, 0<TRUNCATED original_length=1215>`
Variable #24
name: `total_params`, type: `int`
value: `66969621`
Variable #25
name: `total_params_gpt`, type: `int`
value: `81912576`
Variable #26
name: `train_indices`, type: `list`
value: `[14, 3, 35, 31, 28, 17, 13, 69, 11, 54, 4, 78, 71, 27, 29, 64, 68, 77, 12, 45, 41, 44, 34, 26, 79, 75, 37, 74, 51, 0, 48, 10, 58, 66, 21, 52, 9, 73, 60, 6, 5, 24, 40, 22, 36, 16, 2, 65, 7, 32, 61, 33, 47, 43, 20, 19, 67, 18, 53, ...]`
Variable #27
name: `train_texts_gen`, type: `list`
value: `["Complaint: Dana Wu's doctor... under the policy's defined categories.", "Complaint: Dana Wu...ll under eligible recurring bill payments.', 'Complaint: Bob Kim paid $18 in interest on his card ending in 0022 and is confused wh...es are excluded from cashback eligibility.', 'Complaint: Bob Kim used his card ending in 0022 to pay $1,500 in property taxes to th...ts are excluded from cashback eligibility.', 'Complaint: Bob Kim claims his $25 cashback disappeared fro<TRUNCATED original_length=7340>`
